<a href="https://colab.research.google.com/github/thetireddude/self-distillation-lab/blob/main/Apple_Paper_Replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes evaluate
!pip install -q human-eval

In [ ]:
import os
from google.colab import userdata

# Automatically retrieves your token and authenticates the environment
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [ ]:
!hf auth login

In [ ]:
!hf auth whoami

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
EXPERIMENT_NAME = "temp16_topk20_topp08_mbpp_full_strict"

BASE_DIR = f"/content/drive/MyDrive/algoverse_ssd_experiment/{EXPERIMENT_NAME}"

os.makedirs(BASE_DIR, exist_ok=True)

In [ ]:
import subprocess, threading, time

def gpu_monitor(interval=30):
    while True:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True
        )
        print(f"[GPU] {result.stdout.strip()}")
        time.sleep(interval)

threading.Thread(target=gpu_monitor, daemon=True).start()

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
import torch

model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

In [ ]:
import re, json, os, tempfile, subprocess, textwrap, torch
from datasets import load_dataset
from tqdm import tqdm
from torch.utils.data import DataLoader

humaneval = load_dataset("openai/openai_humaneval", split="test")

def extract_code(text):
    # Remove markdown fences if present
    match = re.search(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    if match:
        text = match.group(1)

    # Stop before model-written tests / explanations
    stop_markers = [
        "\nif __name__",
        "\ndef test_",
        "\n# Check",
        "\ncheck_solution",
        "\nprint(",
        "\n```",
        "\nThis Python",
        "\nExplanation",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0]

    return text.rstrip()

def batched_generate_completions(
    model,
    tokenizer,
    problems,
    batch_size=32,
    max_new_tokens=256,
    save_path=None
):
    model.eval()
    completions = {}

    if save_path and os.path.exists(save_path):
        with open(save_path, "r") as f:
            completions = json.load(f)
        print(f"Loaded {len(completions)} existing completions")

    remaining = [p for p in problems if p["task_id"] not in completions]
    print(f"Remaining completions to generate: {len(remaining)}")

    loader = DataLoader(
        remaining,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        collate_fn=lambda batch: batch
    )

    for batch in tqdm(loader, desc="Generating HumanEval completions"):
        prompts = [p["prompt"] for p in batch]
        task_ids = [p["task_id"] for p in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        ).to(model.device)

        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        decoded = tokenizer.batch_decode(
            outputs[:, input_len:],
            skip_special_tokens=True
        )

        for task_id, text in zip(task_ids, decoded):
            completions[task_id] = extract_code(text)

        if save_path:
            with open(save_path, "w") as f:
                json.dump(completions, f, indent=2)

    return completions

def run_humaneval_test(problem, completion, timeout=10):
    code = (
        problem["prompt"]
        + "\n"
        + completion
        + "\n"
        + problem["test"]
        + f"\ncheck({problem['entry_point']})"
    )

    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(code)
        path = f.name

    try:
        result = subprocess.run(
            ["python", path],
            capture_output=True,
            text=True,
            timeout=timeout
        )
        return result.returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.remove(path)

def evaluate_humaneval_fast(
    model,
    tokenizer,
    results_path,
    completions_path,
    batch_size=8,
    max_new_tokens=256,
    timeout=10
):
    problems = list(humaneval)

    completions = batched_generate_completions(
        model=model,
        tokenizer=tokenizer,
        problems=problems,
        batch_size=batch_size,
        max_new_tokens=max_new_tokens,
        save_path=completions_path
    )

    results = []
    passed = 0

    for problem in tqdm(problems, desc="Running HumanEval tests"):
        completion = completions[problem["task_id"]]
        ok = run_humaneval_test(problem, completion, timeout=timeout)

        passed += int(ok)

        results.append({
            "task_id": problem["task_id"],
            "passed": ok,
            "completion": completion
        })

    score = passed / len(problems)

    with open(results_path, "w") as f:
        json.dump({
            "pass_at_1": score,
            "num_passed": passed,
            "num_total": len(problems),
            "results": results
        }, f, indent=2)

    print(f"HumanEval pass@1: {score:.3f} ({passed}/{len(problems)})")

    return score

In [ ]:
base_eval_path = f"{BASE_DIR}/humaneval_base_results.json"
base_completions_path = f"{BASE_DIR}/humaneval_base_completions.json"

base_score = evaluate_humaneval_fast(
    model=model,
    tokenizer=tokenizer,
    results_path=base_eval_path,
    completions_path=base_completions_path,
    batch_size=32,
    max_new_tokens=256,
    timeout=10
)

In [ ]:
import json

completions_path = f"{BASE_DIR}/humaneval_base_completions.json"

with open(completions_path, "r") as f:
    completions = json.load(f)

print(f"Loaded {len(completions)} completions")

for task_id, completion in list(completions.items())[:3]:
    print("=" * 80)
    print("TASK:", task_id)
    print(completion[:2000])

In [ ]:
empty_or_short = {
    task_id: completion
    for task_id, completion in completions.items()
    if len(completion.strip()) < 20
}

print("Very short completions:", len(empty_or_short))
print(list(empty_or_short.items())[:5])

In [ ]:
from datasets import load_dataset

dataset = load_dataset("claudios/google-research-datasets__mbpp", "full", split="train+validation+test")

In [ ]:
len(dataset)

In [ ]:
def make_prompt(example):
    return f"""Write a Python solution for the following programming problem.

Problem:
{example['text']}

Return only the Python code.
"""

# STRICT PROMPT:
#   return f"""Solve the following programming problem:

# Problem:
# {example['text']}


# Output ONLY valid Python code.

# Do not provide explanations.
# Do not provide examples.
# Do not provide markdown.
# Do not provide comments outside the solution
# """

prompts = [make_prompt(x) for x in dataset]

In [ ]:
prompts

In [ ]:
import json, os, torch
from tqdm import tqdm
from torch.utils.data import DataLoader

output_path = f"{BASE_DIR}/ssd_data.jsonl"
batch_size = 64  # start with 4 or 8 on Colab T4; increase if memory allows

# Load completed prompts
generated = {}
if os.path.exists(output_path):
    with open(output_path, "r") as f:
        for line in f:
            row = json.loads(line)
            generated[row["prompt"]] = row

remaining_prompts = [p for p in prompts if p not in generated]

print(f"Already generated: {len(generated)}")
print(f"Remaining: {len(remaining_prompts)}")

prompt_loader = DataLoader(
    remaining_prompts,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

model.eval()

with open(output_path, "a") as f:
    for batch_prompts in tqdm(prompt_loader):
        inputs = tokenizer(
            list(batch_prompts),
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=True,
                temperature=1.6,
                top_k=20,
                top_p=0.8,
                pad_token_id=tokenizer.eos_token_id
            )

        input_len = inputs["input_ids"].shape[1]
        completions = tokenizer.batch_decode(
            outputs[:, input_len:],
            skip_special_tokens=True
        )

        for prompt, completion in zip(batch_prompts, completions):
            row = {
                "prompt": prompt,
                "completion": completion
            }
            f.write(json.dumps(row) + "\n")
            f.flush()

In [ ]:
from datasets import Dataset
import json

output_path = f"{BASE_DIR}/ssd_data.jsonl"

ssd_data = []
with open(output_path, "r") as f:
    for line in f:
        ssd_data.append(json.loads(line))

print(f"Loaded {len(ssd_data)} SSD examples")

train_dataset = Dataset.from_list(ssd_data)

def format_example(example):
    return {
        "text": example["prompt"] + "\n" + example["completion"]
    }

train_dataset = train_dataset.map(format_example)

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

In [ ]:
train_dataset

In [ ]:
train_dataset.column_names

In [ ]:
train_dataset[0]

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=f"{BASE_DIR}/checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    bf16=True,
    fp16=False,
    max_length=1024,
    packing=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=peft_config,
    args=training_args
)

In [ ]:
# use if starting fresh
trainer.train()

In [ ]:
# use if progress was interrupted

trainer.train(resume_from_checkpoint=True)

In [ ]:
trainer.save_model(f"{BASE_DIR}/ssd-qwen-coder-1.5b-lora")

In [ ]:
from peft import PeftModel

adapter_path = f"{BASE_DIR}/ssd-qwen-coder-1.5b-lora"

base_for_eval = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

ssd_model = PeftModel.from_pretrained(
    base_for_eval,
    adapter_path
)

ssd_model.eval()

ssd_eval_path = f"{BASE_DIR}/humaneval_ssd_results.json"
ssd_completions_path = f"{BASE_DIR}/humaneval_ssd_completions.json"

ssd_score = evaluate_humaneval_fast(
    model=ssd_model,
    tokenizer=tokenizer,
    results_path=ssd_eval_path,
    completions_path=ssd_completions_path,
    batch_size=32,
    max_new_tokens=256,
    timeout=10
)

In [ ]:
print("Base HumanEval pass@1:", base_score)
print("SSD HumanEval pass@1:", ssd_score)
print("Delta:", ssd_score - base_score)

In [ ]:
# FOR DEBUGGING:

# import json

# with open(f"{BASE_DIR}/ssd_data.jsonl") as f:
#     examples = [json.loads(next(f)) for _ in range(20)]

# for ex in examples[:5]:
#     print(ex["completion"][:1000])
#     print("=" * 80)

In [ ]:
# FOR DEBUGGING

# import json, random

# ssd_path = f"{BASE_DIR}/ssd_data.jsonl"

# with open(ssd_path, "r") as f:
#     ssd_examples = [json.loads(line) for line in f]

# sample = random.sample(ssd_examples, min(50, len(ssd_examples)))

# for i, ex in enumerate(sample):
#     print("=" * 100)
#     print("EXAMPLE", i)
#     print("PROMPT:")
#     print(ex["prompt"][:500])
#     print("\nCOMPLETION:")
#     print(ex["completion"][:1500])